## Data Foundation

In [29]:
import pandas as pd
import os
from pathlib import Path

# print(os.getcwd())
customers = pd.read_csv('../data/raw/customers.csv')
products = pd.read_csv('../data/raw/products.csv')
orders = pd.read_csv('../data/raw/orders.csv')
tickets = pd.read_csv('../data/raw/tickets.csv',keep_default_na=False)

print("\n========== RAW DATASETS LOADED ==========")

print("Customers:", customers.shape)
print("Products:", products.shape)
print("Orders:", orders.shape)
print("Tickets:", tickets.shape)



========== RAW DATASETS LOADED ==========
Customers: (1200, 5)
Products: (40, 4)
Orders: (13084, 9)
Tickets: (164, 5)


In [30]:
print("\n========== DATASET Dimensions ==========")

datasets = {
    "Customers": customers,
    "Products": products,
    "Orders": orders,
    "Tickets": tickets
}

for name, df in datasets.items():
    rows, cols = df.shape
    print(f"\n{name}")
    print(f"Rows: {rows}")
    print(f"Columns: {cols}")


========== DATASET Dimensions ==========

Customers
Rows: 1200
Columns: 5

Products
Rows: 40
Columns: 4

Orders
Rows: 13084
Columns: 9

Tickets
Rows: 164
Columns: 5


In [31]:
print("\n========== DATA TYPES ==========")

for name, df in datasets.items():

    print(f"\n{name} - Data Types")

    print(df.dtypes)


========== DATA TYPES ==========

Customers - Data Types
customer_id        str
city               str
signup_date        str
age            float64
gender             str
dtype: object

Products - Data Types
product_id        str
product_name      str
category          str
price_inr       int64
dtype: object

Orders - Data Types
order_id                str
customer_id             str
product_id              str
quantity              int64
order_ts                str
status                  str
discount_pct          int64
order_amount            str
delivery_minutes    float64
dtype: object

Tickets - Data Types
ticket_id        str
customer_id      str
order_id         str
text             str
true_category    str
dtype: object


In [32]:
print("\n========== MISSING VALUE AUDIT ==========")

for name, df in datasets.items():

    print(f"\n{name} - Missing Values")

    missing_count = df.isna().sum()

    missing_pct = df.isna().mean() * 100

    missing_report = pd.DataFrame({
        "missing_count": missing_count,
        "missing_pct": missing_pct.round(2)
    })

    print(missing_report)



========== MISSING VALUE AUDIT ==========

Customers - Missing Values
             missing_count  missing_pct
customer_id              0         0.00
city                     0         0.00
signup_date              0         0.00
age                     61         5.08
gender                  30         2.50

Products - Missing Values
              missing_count  missing_pct
product_id                0          0.0
product_name              0          0.0
category                  0          0.0
price_inr                 0          0.0

Orders - Missing Values
                  missing_count  missing_pct
order_id                      0         0.00
customer_id                   0         0.00
product_id                    0         0.00
quantity                      0         0.00
order_ts                      0         0.00
status                        0         0.00
discount_pct                  0         0.00
order_amount                 53         0.41
delivery_minutes           

In [33]:
print("\n========== CUSTOMER-SPECIFIC AUDIT ==========")

age_numeric = pd.to_numeric(
    customers["age"],
    errors="coerce"
)

age_text = customers["age"].astype("string").str.strip()

blank_ages = age_text.isna() | age_text.eq("")

unparseable_ages = (
    ~blank_ages &
    age_numeric.isna()
)

invalid_age_range = (
    age_numeric.notna() &
    ~age_numeric.between(10, 100)
)

print("Blank ages:", int(blank_ages.sum()))

print("Unparseable ages:", int(unparseable_ages.sum()))

print("Ages outside 10-100:", int(invalid_age_range.sum()))

print(
    "Total blank or invalid ages:",
    int((blank_ages | unparseable_ages | invalid_age_range).sum())
)


========== CUSTOMER-SPECIFIC AUDIT ==========
Blank ages: 61
Unparseable ages: 0
Ages outside 10-100: 6
Total blank or invalid ages: 67


In [34]:
print("\nRaw city spellings:")

print(
    customers["city"]
    .value_counts(dropna=False)
)


Raw city spellings:
city
Pune           148
Delhi          123
Hyderabad      119
Bengaluru      115
Mumbai         113
Chennai        106
chennai         44
hyderabad       40
delhi           36
mumbai          35
pune            35
BENGALURU       32
 Delhi          30
bengaluru       27
PUNE            24
 Bengaluru      23
DELHI           21
CHENNAI         21
 Chennai        20
 Pune           20
 Mumbai         19
 Hyderabad      18
MUMBAI          17
HYDERABAD       14
Name: count, dtype: int64


In [35]:
print("\nRaw gender spellings:")

print(
    customers["gender"]
    .value_counts(dropna=False)
)



Raw gender spellings:
gender
female    136
FEMALE    122
F         120
m         118
f         117
Female    115
male      114
MALE      110
M         110
Male      108
NaN        30
Name: count, dtype: int64


In [36]:
print("\n========== ORDER-SPECIFIC AUDIT ==========")

exact_duplicates = orders.duplicated().sum()

print("Exact duplicate rows:", int(exact_duplicates))

pending_records = (
    orders["status"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("pending")
    .sum()
)

print("Pending records:", int(pending_records))

distinct_order_ids = orders["order_id"].nunique()

print("Distinct order IDs:", int(distinct_order_ids))
amount_text = orders["order_amount"].astype("string").str.strip()

amount_numeric = pd.to_numeric(
    orders["order_amount"],
    errors="coerce"
)

blank_amounts = (
    amount_text.isna() |
    amount_text.eq("")
)

unparseable_amounts = (
    ~blank_amounts &
    amount_numeric.isna()
)

print("Blank order amounts:", int(blank_amounts.sum()))

print(
    "Unparseable order amounts:",
    int(unparseable_amounts.sum())
)

delivery_text = (
    orders["delivery_minutes"]
    .astype("string")
    .str.strip()
)

delivery_numeric = pd.to_numeric(
    orders["delivery_minutes"],
    errors="coerce"
)

blank_delivery = (
    delivery_text.isna() |
    delivery_text.eq("")
)

unparseable_delivery = (
    ~blank_delivery &
    delivery_numeric.isna()
)

negative_delivery = (
    delivery_numeric.notna() &
    delivery_numeric.lt(0)
)

print("Blank delivery times:", int(blank_delivery.sum()))

print(
    "Unparseable delivery times:",
    int(unparseable_delivery.sum())
)

print(
    "Negative delivery times:",
    int(negative_delivery.sum())
)



========== ORDER-SPECIFIC AUDIT ==========
Exact duplicate rows: 119
Pending records: 656
Distinct order IDs: 12309
Blank order amounts: 53
Unparseable order amounts: 67
Blank delivery times: 2541
Unparseable delivery times: 0
Negative delivery times: 106


In [37]:
customers_clean = customers.copy()

customers_clean["city"] = (customers_clean["city"].astype("string")
    .str.strip())

customers_clean["city"] = (customers_clean["city"].str.title())

valid_cities = ["Chennai", "Bengaluru", "Mumbai", "Delhi", "Hyderabad", "Pune"]

unexpected_cities = (customers_clean.loc[~customers_clean["city"].isin(valid_cities), "city"].drop_duplicates().to_list())

print("\nUnexpected city values:", unexpected_cities)

assert len(unexpected_cities) == 0, (
    f"Unexpected city names found: {unexpected_cities}"
)

print("Unique cities:", customers_clean["city"].nunique())

print(customers_clean["city"].value_counts())


Unexpected city values: []
Unique cities: 6
city
Pune         227
Delhi        210
Bengaluru    197
Chennai      191
Hyderabad    191
Mumbai       184
Name: count, dtype: Int64


In [38]:
gender_normalized = customers_clean["gender"].astype("string").str.strip().str.lower()

gender_mapping = {
    "m": "male",
    "male": "male",
    "f": "female",
    "female": "female","": "unknown"}

unexpected_genders = (gender_normalized[gender_normalized.notna()& ~gender_normalized.isin(gender_mapping.keys())].drop_duplicates().to_list())

print("\nUnexpected gender values:", unexpected_genders)

customers_clean["gender"] = (gender_normalized.map(gender_mapping).fillna("unknown"))

print(customers_clean["gender"].value_counts())




Unexpected gender values: []
gender
female     610
male       560
unknown     30
Name: count, dtype: int64


In [39]:
customers_clean["age"] = pd.to_numeric(
    customers_clean["age"],
    errors="coerce"
)

print(customers_clean["age"].dtype)

invalid_age_mask = ~customers_clean["age"].between(10, 100)

invalid_age_count = int(invalid_age_mask.sum())

print("Invalid age count:", invalid_age_count)

customers_clean.loc[invalid_age_mask, "age"] = float("nan")

city_median = customers_clean.groupby("city")["age"].transform("median")

customers_clean["age"] = customers_clean["age"].fillna(city_median)

print("Missing ages after imputation:", int(customers_clean["age"].isna().sum()))

assert customers_clean["age"].between(10, 100).all(), (
    "Some ages are still missing or outside the permitted range."
)

print("All customer ages are valid.")

float64
Invalid age count: 67
Missing ages after imputation: 0
All customer ages are valid.


In [40]:
customers_clean["signup_date"] = pd.to_datetime(customers_clean["signup_date"], errors="raise")
print(customers_clean["signup_date"].dtype)

datetime64[us]


In [41]:
print("Total Customers:",len(customers_clean))
print("number of unique cities:", customers_clean["city"].nunique())
print("\n Gender distribution:")
print(customers_clean["gender"].value_counts().reindex(["female","male","unknown"], fill_value=0))

print("\n missing ages:",customers_clean["age"].isna().sum())

print("Age range:",customers_clean["age"].min(),"-",customers_clean["age"].max())

print("\n signup date data type:",customers_clean["signup_date"].dtype)

assert len(customers_clean) == 1200

assert customers_clean["city"].nunique() == 6

assert customers_clean["gender"].value_counts().to_dict() == {"female": 610, "male": 560, "unknown": 30}

assert customers_clean["age"].notna().all()

assert customers_clean["age"].between(10, 100).all()

assert pd.api.types.is_datetime64_any_dtype(customers_clean["signup_date"])

print("\n customers cleaning checkpoints passed")

Total Customers: 1200
number of unique cities: 6

 Gender distribution:
gender
female     610
male       560
unknown     30
Name: count, dtype: int64

 missing ages: 0
Age range: 18.0 - 60.0

 signup date data type: datetime64[us]

 customers cleaning checkpoints passed


In [42]:
orders_clean = orders.copy()

orders_clean["order_ts"] = pd.to_datetime(orders_clean["order_ts"], errors="raise")

print("Order timestamp type:", orders_clean["order_ts"].dtype)

Order timestamp type: datetime64[us]


In [43]:
duplicates_before = orders_clean.duplicated().sum()

print("Exact duplicate rows before cleaning:", int(duplicates_before))

orders_clean = orders_clean.drop_duplicates().copy()

duplicates_after = orders_clean.duplicated().sum()

print("Exact duplicate rows after cleaning:", int(duplicates_after))

assert duplicates_after == 0

orders_clean = orders_clean.sort_values(by="order_ts", ascending=True).copy()

orders_clean = orders_clean.drop_duplicates(subset=["order_id"], keep="last").copy()

orders_clean = orders_clean.reset_index(drop=True)

print("Distinct order IDs after cleaning:", int(orders_clean["order_id"].nunique()))

print("remaining duplicate order ids:", orders_clean["order_id"].duplicated().sum())

Exact duplicate rows before cleaning: 119
Exact duplicate rows after cleaning: 0
Distinct order IDs after cleaning: 12309
remaining duplicate order ids: 0


In [44]:
product_details = products[["product_id", "category", "price_inr"]].copy()

product_details["price_inr"] = pd.to_numeric(product_details["price_inr"], errors="raise")

orders_clean = orders_clean.merge(product_details, on="product_id", how="left", validate = "many_to_one")

assert orders_clean["category"].notna().all(),("some orders do not have a matching product.")

assert orders_clean["price_inr"].notna().all(),("some orders do not have a matching product price.")
print("Orders with product information:", len(orders_clean))

Orders with product information: 12309


In [45]:
orders_clean["order_amount"] = pd.to_numeric(orders_clean["order_amount"], errors="coerce")

print("Order amount data type:", orders_clean["order_amount"].dtype)

print("Missing amounts after conversion:", orders_clean["order_amount"].isna().sum())

missing_amount_mask = orders_clean["order_amount"].isna()
missing_amount_count = int(missing_amount_mask.sum())
print("\nMissing amounts before calculation:", missing_amount_count)

calculated_amount = orders_clean["price_inr"] * orders_clean["quantity"] * (1 - orders_clean["discount_pct"] / 100).round(2)

orders_clean.loc[missing_amount_mask, "order_amount"] = calculated_amount[missing_amount_mask]

print("Missing amounts after calculation:", orders_clean["order_amount"].isna().sum())

assert orders_clean["order_amount"].notna().all(),("some orders still have missing order amounts.")

print("All orders have valid order amounts.")

Order amount data type: float64
Missing amounts after conversion: 112

Missing amounts before calculation: 112
Missing amounts after calculation: 0
All orders have valid order amounts.


In [46]:
orders_clean["delivery_minutes"] = pd.to_numeric(orders_clean["delivery_minutes"], errors="coerce")

negative_delivery_mask = orders_clean["delivery_minutes"] < 0

negative_delivery_count = int(negative_delivery_mask.sum())

print("\n Negative delivery times found:", negative_delivery_count)

orders_clean.loc[negative_delivery_mask, "delivery_minutes"] = float("nan")

print("Negative delivery times replaced with missing values.")


 Negative delivery times found: 105
Negative delivery times replaced with missing values.


In [47]:
completed_mask = orders_clean["status"].eq("completed")

category_median = (orders_clean.loc[completed_mask].groupby("category")["delivery_minutes"].transform("median"))

orders_clean.loc[completed_mask, "delivery_minutes"] = (orders_clean.loc[completed_mask, "delivery_minutes"].fillna(category_median))


In [48]:
completed_missing_delivery = (
    orders_clean.loc[
        orders_clean["status"].eq("completed"),
        "delivery_minutes"
    ].isna().sum()
)

cancelled_missing_delivery = (
    orders_clean.loc[
        orders_clean["status"].eq("cancelled"),
        "delivery_minutes"
    ].isna().sum()
)

print(
    "\nMissing delivery times in completed orders:",
    completed_missing_delivery
)

print(
    "Missing delivery times in cancelled orders:",
    cancelled_missing_delivery
)


Missing delivery times in completed orders: 0
Missing delivery times in cancelled orders: 1199


In [49]:
print("\nOrder status distribution:")

print(orders_clean["status"].value_counts())

print("\n unique orders:",orders_clean["order_id"].nunique())

completed_count = int(orders_clean["status"].eq("completed").sum())
cancelled_count = int(orders_clean["status"].eq("cancelled").sum())
pending_count = int(orders_clean["status"].eq("pending").sum())

print("\nCompleted orders:", completed_count)

print("Cancelled orders:", cancelled_count)

print("Pending orders:", pending_count)

missing_amount_count = int(orders_clean["order_amount"].isna().sum())

print("\nMissing order amounts:", missing_amount_count)

completed_missing = int(
    orders_clean.loc[
        orders_clean["status"].eq("completed"),
        "delivery_minutes"
    ].isna().sum()
)

print("Missing completed delivery times:", completed_missing)

cancelled_missing = int(
    orders_clean.loc[
        orders_clean["status"].eq("cancelled"),
        "delivery_minutes"
    ].isna().sum()
)

print("Missing cancelled delivery times:",cancelled_missing)

completed_revenue = orders_clean.loc[
    orders_clean["status"].eq("completed"),
    "order_amount"
].sum()

print("\nCompleted-order revenue: INR",f"{completed_revenue:,.2f}")



Order status distribution:
status
completed    11110
cancelled     1199
Name: count, dtype: int64

 unique orders: 12309

Completed orders: 11110
Cancelled orders: 1199
Pending orders: 0

Missing order amounts: 0
Missing completed delivery times: 0
Missing cancelled delivery times: 1199

Completed-order revenue: INR 2,998,668.80


In [50]:
assert len(orders_clean) == 12309

assert orders_clean["order_id"].nunique() == 12309

assert orders_clean["order_id"].duplicated().sum() == 0

assert completed_count == 11110

assert cancelled_count == 1199

assert pending_count == 0

assert missing_amount_count == 0

assert completed_missing == 0

assert cancelled_missing == 1199

assert round(completed_revenue, 2) == 2998668.80


In [54]:
path = Path('../data/clean')
print(path)

path.mkdir(parents=True, exist_ok=True)
customers_clean.to_csv(path/'customers_clean.csv', index=False)

orders_clean["order_ts"] = pd.to_datetime(orders_clean["order_ts"], errors="coerce")
orders_clean["order_ts"] = orders_clean["order_ts"].dt.strftime('%Y-%m-%d %H:%M:%S')
orders_clean.to_csv(path/'orders_clean.csv', index=False)
products.to_csv(path/'products_clean.csv', index=False)

print("\n all 3 dataframes saved as csv files successfully in the clean folder")


..\data\clean

 all 3 dataframes saved as csv files successfully in the clean folder


In [56]:
# Check whether the files exist
print(list(path.iterdir()))

[WindowsPath('../data/clean/customers_clean.csv'), WindowsPath('../data/clean/orders_clean.csv'), WindowsPath('../data/clean/products_clean.csv')]


In [58]:
customers_check = pd.read_csv("../data/clean/customers_clean.csv")
products_check = pd.read_csv("../data/clean/products_clean.csv")
orders_check = pd.read_csv("../data/clean/orders_clean.csv")

print(customers_check.shape)
print(products_check.shape)
print(orders_check.shape)

(1200, 5)
(40, 4)
(12309, 11)


In [61]:
assert customers_check.shape == customers_clean.shape
assert products_check.shape == products.shape
assert orders_check.shape == orders_clean.shape

print("CSV validation successful!")

CSV validation successful!


In [65]:

import sqlite3
from pathlib import Path

# Step 1: Define the database path
db_path = Path("../data/instamart.db")

# Step 2: Ensure the parent directory exists
db_path.parent.mkdir(parents=True, exist_ok=True)

# Step 3: Establish the SQLite connection
conn = sqlite3.connect(db_path)

try:

    # Step 4: Insert customers DataFrame
    customers.to_sql(
        "customers",
        conn,
        if_exists="replace",
        index=False
    )

    # Step 5: Insert products DataFrame
    products.to_sql(
        "products",
        conn,
        if_exists="replace",
        index=False
    )

    # Step 6: Insert orders DataFrame
    orders.to_sql(
        "orders",
        conn,
        if_exists="replace",
        index=False
    )

    # Step 7: Commit the changes
    conn.commit()

    print("All three tables successfully created!")

finally:

    # Step 8: Close the connection
    conn.close()


All three tables successfully created!


In [66]:
conn = sqlite3.connect("../data/instamart.db")

In [67]:
query = """
SELECT name
FROM sqlite_master
WHERE type = 'table';
"""

tables = pd.read_sql_query(query, conn)

print(tables)

        name
0  customers
1   products
2     orders


In [68]:
customer_count = pd.read_sql_query(
    "SELECT COUNT(*) AS total FROM customers",
    conn
)

product_count = pd.read_sql_query(
    "SELECT COUNT(*) AS total FROM products",
    conn
)

order_count = pd.read_sql_query(
    "SELECT COUNT(*) AS total FROM orders",
    conn
)

print("Customers:", customer_count.iloc[0, 0])
print("Products:", product_count.iloc[0, 0])
print("Orders:", order_count.iloc[0, 0])

Customers: 1200
Products: 40
Orders: 13084


In [69]:
assert customer_count.iloc[0, 0] == len(customers)

assert product_count.iloc[0, 0] == len(products)

assert order_count.iloc[0, 0] == len(orders)

print("Row count validation successful!")

Row count validation successful!


In [70]:
for table in ["customers", "products", "orders"]:

    print(f"\nSchema of {table}:")

    schema = pd.read_sql_query(
        f"PRAGMA table_info({table})",
        conn
    )

    print(schema[["name", "type"]])


Schema of customers:
          name  type
0  customer_id  TEXT
1         city  TEXT
2  signup_date  TEXT
3          age  REAL
4       gender  TEXT

Schema of products:
           name     type
0    product_id     TEXT
1  product_name     TEXT
2      category     TEXT
3     price_inr  INTEGER

Schema of orders:
               name     type
0          order_id     TEXT
1       customer_id     TEXT
2        product_id     TEXT
3          quantity  INTEGER
4          order_ts     TEXT
5            status     TEXT
6      discount_pct  INTEGER
7      order_amount     TEXT
8  delivery_minutes     REAL


In [71]:
sample_orders = pd.read_sql_query(
    """
    SELECT order_id, order_ts
    FROM orders
    LIMIT 5
    """,
    conn
)

print(sample_orders)

     order_id             order_ts
0  ORD-108250  2025-09-01 12:24:59
1  ORD-101939  2025-08-17 07:07:23
2  ORD-108765  2025-07-13 09:42:32
3  ORD-108861  2025-07-04 11:23:24
4  ORD-109150  2025-05-07 15:35:01


In [72]:
timestamp_type = pd.read_sql_query(
    """
    SELECT typeof(order_ts) AS storage_type
    FROM orders
    LIMIT 5
    """,
    conn
)

print(timestamp_type)

  storage_type
0         text
1         text
2         text
3         text
4         text


In [73]:
invalid_timestamps = pd.read_sql_query(
    """
    SELECT COUNT(*) AS invalid_count
    FROM orders
    WHERE order_ts IS NULL
       OR strftime('%Y-%m-%d %H:%M:%S', order_ts) IS NULL
       OR strftime('%Y-%m-%d %H:%M:%S', order_ts) != order_ts
    """,
    conn
)

print(invalid_timestamps)

   invalid_count
0              0


In [74]:
conn.close()